In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf
import gspread
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

In [20]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"
target_sheet_url = "https://docs.google.com/spreadsheets/d/1OU786oBbozzYkvr2O9WUG3IECJfLgxVwDOt_lZRsAKo/edit?gid=661065461#gid=661065461"

In [21]:

def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)
        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")

In [ ]:
target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA',  'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA',  
    'RINSIYA', 'ARSHAD','NAFI', 'SHIHAD' ,'RAHIYAD'
]

# --- BDM / TEAM MAPPING ---
# Define your teams here. Ensure names match exactly what is in the sheets (UPPERCASE).
TEAM_MAP = {
    'RAHIB': 'SHAHABAS TEAM',
    'HABIYA': 'SHAHABAS TEAM',
    'BURHANA': 'SHAHABAS TEAM',
    'SHAMNA': 'SHAHABAS TEAM',
    'ARUN': 'SHAHABAS TEAM',
    'CHAITHANYA': 'SHAHABAS TEAM',
    'ZAKIYA': 'MANASA TEAM',
    'SAFAN': 'MANASA TEAM',
    'SUHANTHIKA': 'MANASA TEAM',
    'ADWAITHA': 'MANASA TEAM',
    'ARSHAD': 'MANASA TEAM',
    'NEHA': 'JEFFIN TEAM',
    'GOWTHAM': 'JEFFIN TEAM',
    'AMINA': 'JEFFIN TEAM',
    'RINSIYA': 'JEFFIN TEAM',
    'NAFI': 'JEFFIN TEAM',
    # 'BURHANA': 'SHAHABAS TEAM',
    # 'SHAMNA': 'SHAHABAS TEAM',
    
}

lead_statuses = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]
potential_targets = ["SUPER HOT", "HOT", "WARM"]

# --- 2. INITIALIZE CONNECTION ---
try:
    c = conf.get_config(file_name=config_path)
    emarath_spread = Spread(emarath_global_sheet_url, config=c)
    target_spread = Spread(target_sheet_url, config=c)
    print("Successfully connected to Google Sheets.")
except Exception as e:
    print(f"Authentication Error: {e}")

# --- 3. DATA EXTRACTION & CLEANING ---
all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'AGENT'        : 'AGENT',
            'COUNTRY'      : 'REGION',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'PRODUCT 1'    : 'PRODUCT',
            'NAME'         : 'NAME',
            'PHONE NO 1'   : 'PHONE NO',
            'STATUS'       : 'STATUS',
            'DATE'         : 'DATE',
            'CALL STATUS'  : 'CALL STATUS'
        }
        
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True).dt.date
            subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
            subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
            
            invalid_vals = ['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0']
            subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(invalid_vals)]

            if not subset.empty:
                for col in ['STATUS', 'AGENT', 'CALL STATUS']:
                    subset[col] = subset[col].astype(str).str.strip().str.upper()
                
                # Apply the Team Mapping
                subset['BDM_TEAM'] = subset['AGENT'].map(TEAM_MAP).fillna('OTHER')
                
                all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error processing sheet '{sheet_name}': {e}")

# --- 4. REPORT GENERATION ---
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    
    # A. Grouping including BDM_TEAM
    group_cols = ['DATE', 'BDM_TEAM', 'AGENT']
    total_leads = master_df.groupby(group_cols)['PHONE NO'].count().to_frame('TOTAL LANDED LEADS')

    # B. Status Pivot Table
    status_pivot = master_df.pivot_table(
        index=group_cols, 
        columns='STATUS', 
        aggfunc='size', 
        fill_value=0
    )
    status_pivot = status_pivot.add_suffix('_S')

    # C. Join and Clean
    final_report = total_leads.join(status_pivot, how='left').fillna(0).astype(int)
    final_report.columns = [col.replace('_S', '') if col.endswith('_S') else col for col in final_report.columns]

    # D. Potential Lead Calculation
    present_potentials = [c for c in potential_targets if c in final_report.columns]
    final_report['POTENTIAL LEAD'] = final_report[present_potentials].sum(axis=1) if present_potentials else 0

    # E. Formatting and Flattening
    final_report = final_report.reset_index()
    
    # Reorder columns to put BDM_TEAM near the front
    ordered_statuses = [s for s in lead_statuses if s in final_report.columns]
    front_cols = ['DATE', 'BDM_TEAM', 'AGENT', 'TOTAL LANDED LEADS', 'POTENTIAL LEAD']
    final_cols = front_cols + [s for s in ordered_statuses if s not in front_cols]
    
    final_report = final_report.reindex(columns=final_cols)
    final_report = final_report.sort_values(by=['DATE', 'BDM_TEAM', 'AGENT'], ascending=[True, True, True])

    # --- 5. OUTPUT ---
    print(f"\nFinal Flattened Report with BDM Mapping:")
    safe_upload(target_spread, final_report, "DATA")
    display(final_report)

else:
    print("No valid lead data found.")

Successfully connected to Google Sheets.

Final Flattened Report with BDM Mapping:


,DATE,BDM_TEAM,AGENT,TOTAL LANDED LEADS,POTENTIAL LEAD,WON,SUPER HOT,HOT,WARM,COLD,BOOKING,WHATS APP ENGAGE
0,2026-02-04,JEFFIN TEAM,AMINA,36,4,2,0,2,2,1,0,28
1,2026-02-04,JEFFIN TEAM,NEHA,27,6,1,1,1,4,2,0,0
2,2026-02-04,SHAHABAS TEAM,ARUN,34,0,1,0,0,0,25,0,8
3,2026-02-04,SHAHABAS TEAM,BURHANA,30,4,1,1,1,2,23,0,2
4,2026-02-04,SHAHABAS TEAM,CHAITHANYA,41,7,3,4,0,3,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
248,2026-02-28,SHAHABAS TEAM,CHAITHANYA,38,3,4,0,0,3,30,1,0
249,2026-02-28,SHAHABAS TEAM,HABIYA,35,0,15,0,0,0,18,0,1
250,2026-02-28,SHAHABAS TEAM,RAHIB,37,1,10,1,0,0,25,1,0
251,2026-03-02,JEFFIN TEAM,RINSIYA,3,0,1,0,0,0,1,0,0
